# Evaluating the full pipeline: candidate generation and reranking

Runs the full inference pipeline of the proposed model on a Karpathy split (test or val) and evaluates it at the points
reported in the paper:

| Point | How the caption is chosen |
|---|---|
| `before_rerank` | first candidate (plain beam search), not reranked |
| `three_score` | three inherited signals: 0.2·S_base + 0.4·S_align + 0.4·S_det |
| `plus_s_retr` | three signals + 0.1·S_retr |
| `plus_s_obj` | three signals + 1·S_obj (threshold r = 3) |
| `five_score` | **five signals**, S_obj with r = 3 (main configuration of the paper) |
| `five_score_1of5` | five signals, S_obj with r = 1 (to compare thresholds) |

Steps for each image:

1. **Evidence** E(I) = YOLO-World (threshold 0.25) ∪ objects named by ≥ r of the K = 5 retrieved captions. The retrieved
   captions come from the training-set caption pool, ranked by CLIP ViT-B/32 cosine similarity.
2. **Fixed blocking rule** during decoding: subtract δ from the logits of COCO object words not in E(I). `RUN_BOTH = True`
   runs the full pipeline twice, without blocking (δ = 0) and with blocking (δ = `DELTA`), saving one file per δ. `RUN_BOTH = False` only
   runs `DELTA` (set `DELTA = 0` to run without blocking only). Running both takes twice as long. `EVIDENCE_SOURCES`
   picks the evidence source for the blocking rule and S_obj at δ = `DELTA`: `"union"` (proposed model), `"yolo"` (YOLO-World only,
   C8), `"retr"` (retrieved captions only, C9). Each source is a separate generation pass and a separate file (`..._d3_yolo_test.json`).
3. **Generate 5 candidates**: 1 plain beam-search caption (beam 3) plus diverse beam search (10 beams, 5 groups, diversity
   penalty 0.8).
4. **Three inherited signals** (formula reranker): S_base is the mean log-probability, S_align is CLIPScore
   (CLIP ViT-B/16), S_det is belief revision against the YOLO objects.
5. **Two proposed signals:**
   - S_obj = −(number of objects the caption names that are not in E(I)), where E(I) uses YOLO-World at threshold `SOBJ_CONF` = 0.55
     (chosen on val). The blocking rule and S_det still use threshold 0.25.
   - S_retr = mean cosine similarity between the caption and the K retrieved captions.
6. **Aggregation:** each signal is min-max normalised over the image's five candidates, the signals are summed with their weights, and the
   caption with the highest score is selected.

**Configuration** (CONFIG cell): `CHECKPOINT_NAME`, `SPLIT`, `DELTA`; weights `W_BASE/W_ALIGN/W_DET`,
`W_RETR`, `W_OBJ`; `SOBJ_CONF`; `COMPUTE_SPICE` (needs Java 8, slow). The default is the δ = 3 blocking rule +
reranking, the configuration of the paper's tables. `CHECKPOINT_NAME` is the epoch-40 checkpoint of one seed (41, 42, 43):
`seed<N>_dynlogits_contrastive_attnpool_epoch40_model.pth` (proposed model) or `baseline_epoch40_seed<N>.pth`
(CE baseline).

**Metrics:** BLEU-1..4, METEOR, ROUGE-L, CIDEr (`simple` tokenisation), optional SPICE, CHAIRs, CHAIRi. The CHAIR ground truth
is the union of the objects in the COCO instance labels and the objects named in the 5 reference captions.

**Saved output:** a JSON file with `metrics` for each point and `rows` in the repository's report format (candidates, ranking,
YOLO objects), so the scripts in `eval/` can reuse it. The generation loop saves progress and
can resume if interrupted.

**Environment:** the notebook calls the repository's root modules directly (no copies), so the numbers match
`retrieval_logit_bias_experiment.py`, `logit_bias_final_eval.py`, `signal_table.py` and
`sobj_threshold_eval.py` in `eval/`. It needs `torch`, `transformers`, `ultralytics`, `pycocoevalcap`, `nltk`, and Java for METEOR.

**Running on Kaggle** (T4 or P100 GPU, Internet on). Attach:
- COCO 2014 (`val2014` images, `instances_val2014.json`) and the Karpathy split (`dataset_coco.json`), as in the training notebooks;
- a self-uploaded code + weights dataset with the repository's root modules (`local_captioner.py`, `generate.py`,
  `clipscore.py`, `formula_reranker.py`, `chair_metric.py`, `chair_synonyms.txt`, `yolo_objects.py`, `vocab.py`,
  `schema.py`, `evaluate_reranked_quality.py`, all in one folder), the vocabulary file
  and `yolov8l-worldv2.pt`;
- the checkpoint to evaluate, attached separately (dataset or Kaggle Model) so it is easy to swap, with a file name matching `CHECKPOINT_NAME`. You can also add `train_captions.json` + `train_caption_emb_b32.npy` to skip re-encoding the retrieval corpus.

The notebook finds files by name under `/kaggle/input` and writes results to `/kaggle/working/results/`. Do a test run with
`MAX_IMAGES = 5` first.

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
import json, os, shutil, sys, time
from collections import Counter
from pathlib import Path

# Model to evaluate. CE baseline: dynlogits_contrastive_attnpool_epoch40_model.pth; proposed DL+CL model:
# dynlogits_contrastive_attnpool_epoch40_model.pth. Both share the vocabulary below.
# Give the full path under /kaggle/input or just the file name (the notebook searches every attached input).
CHECKPOINT_NAME = "seed41_dynlogits_contrastive_attnpool_epoch40_model.pth"
VOCAB_NAME = "coco2014_karpathy_restval_dynlogits_contrastive_attnpool_vocab.json"
YOLO_NAME = "yolov8l-worldv2.pt"   # YOLO-World

# The notebook finds files by name in every dataset attached under /kaggle/input, so it does not depend on dataset names.
FILES = {}
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d not in ("train2014", "test2014")]   # skip unused images for speed
    for f in files:
        FILES.setdefault(f, Path(root) / f)


def find(name):
    """Use a full path as is; otherwise search by file name."""
    p = Path(name)
    if p.is_file():
        return p
    assert p.name in FILES, f"{p.name} not found in /kaggle/input, attach the dataset/model that contains it"
    return FILES[p.name]


CODE_DIR = find("local_captioner.py").parent        # code dataset: the repository's root .py modules
KARPATHY_JSON = find("dataset_coco.json")
INSTANCES_JSON = find("instances_val2014.json")
# Karpathy val/test images all belong to val2014; take the folder with the most images to avoid picking a stray sample folder
IMAGE_DIR = Counter(p.parent for n, p in FILES.items() if n.startswith("COCO_val2014_")).most_common(1)[0][0]
# Checkpoint attached separately (dataset or Kaggle Model); to change the model, attach the new one and edit CHECKPOINT_NAME
CHECKPOINT, VOCAB_FILE, YOLO_MODEL = find(CHECKPOINT_NAME), find(VOCAB_NAME), find(YOLO_NAME)
# Pre-encoded retrieval corpus (train_captions.json + train_caption_emb_b32.npy): used if attached, otherwise encoded here
# (only accept a folder that has both files, since some datasets only contain a copy of the .npy file)
RETRIEVAL_CACHE = next((p.parent for p in [FILES.get("train_captions.json")] if p and (p.parent / "train_caption_emb_b32.npy").is_file()),
                       Path("/kaggle/working/retrieval_cache"))
RESULTS_DIR = Path("/kaggle/working/results")

SPLIT = "test"        # "test" | "val"
START, MAX_IMAGES = 0, None   # images sorted by file name as in the paper's reports; e.g. MAX_IMAGES = 5 for a test run
DELTA = 3.0           # strength of the fixed blocking rule; 0 = no blocking

YOLO_CONF, YOLO_MAX_OBJECTS = 0.25, 20   # threshold for the blocking rule and S_det
K_RETR = 5            # number of retrieved captions
MAJ = 3               # majority threshold r: an object named by >= r retrieved captions enters the evidence set
MAX_NEW_TOKENS = 40
FORCE_CPU = False     # True: do not use the GPU

# Reranker (values fixed on the validation split, Section 4.2 of the paper)
W_BASE, W_ALIGN, W_DET = 0.2, 0.4, 0.4
W_RETR, W_OBJ = 0.1, 1.0
SOBJ_CONF = 0.55          # separate YOLO-World threshold for the S_obj evidence set (chosen on val); the blocking rule and S_det still use YOLO_CONF
CLIP_SCORER = "openai/clip-vit-base-patch16"
COMPUTE_SPICE = False      # True: add SPICE for before_rerank, three_score, five_score (Java 8, about 1 hour per 5,000 images)
SAVE_EVERY = 100           # save progress every this many images; re-running the generation cell continues where it stopped
RUN_BOTH = True            # True: run the full pipeline twice, without blocking (δ = 0) then with blocking (δ = DELTA); twice the time
# Evidence source for the blocking rule and S_obj at δ = DELTA: "union" (YOLO-World ∪ retrieval, proposed model), "yolo" (only
# YOLO-World, C8), "retr" (retrieved captions only, C9). Each source is a separate generation pass; δ = 0 has no blocking, so only "union" runs.
EVIDENCE_SOURCES = ["union"]
RUNS = list(dict.fromkeys(([(0.0, "union")] if RUN_BOTH else []) + [(DELTA, src) for src in EVIDENCE_SOURCES]))


def output_of(run):
    delta, src = run
    return RESULTS_DIR / f"eval_rerank_{CHECKPOINT.stem}_d{delta:g}{'' if src == 'union' else '_' + src}_{SPLIT}.json"


def label(run):
    return f"δ = {run[0]:g}" + ("" if run[1] == "union" else f", evidence {run[1]}")

In [ ]:
# ==========================================
# LOAD REPOSITORY CODE, DATA AND CHAIR LABELS
# ==========================================
import subprocess   # requires Internet to be enabled in the notebook settings
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pycocoevalcap", "python-dotenv"], check=True)
if shutil.which("java") is None:   # METEOR needs Java
    subprocess.run("apt-get -qq update && apt-get -qq install -y default-jre-headless > /dev/null", shell=True)
import nltk
for pkg in ("punkt", "punkt_tab", "wordnet", "omw-1.4"):
    nltk.download(pkg, quiet=True)
USE_METEOR = shutil.which("java") is not None
if not USE_METEOR:
    print("[!] Java not found: skipping METEOR")
if FORCE_CPU:
    import torch
    torch.cuda.is_available = lambda: False   # the repository modules pick their device with this function
sys.path.insert(0, str(CODE_DIR))

import numpy as np
import torch
from tqdm.auto import tqdm

import chair_metric
import local_captioner
import vocab as vocab_mod
from evaluate_reranked_quality import compute_metrics
from PIL import Image
from yolo_objects import YoloObjectExtractor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
data = json.loads(KARPATHY_JSON.read_text(encoding="utf-8"))
items = sorted(({"image": im["filename"], "references": [s["raw"] for s in im["sentences"]]}
                for im in data["images"] if im["split"] == SPLIT), key=lambda r: r["image"])
items = items[START: START + MAX_IMAGES] if MAX_IMAGES else items[START:]
paths = [IMAGE_DIR / r["image"] for r in items]
missing = [p.name for p in paths if not p.is_file()]
assert not missing, f"{len(missing)} images missing, e.g. {missing[:3]}"

OBJECTS, INV = chair_metric.load_synonyms()
fn2id = chair_metric.load_filename_to_cocoid(KARPATHY_JSON)
inst = chair_metric.load_instance_objects_by_cocoid(INSTANCES_JSON, INV)
# CHAIR ground truth = objects in COCO instances  U  objects named in the 5 reference captions
GT = {r["image"]: chair_metric.ground_truth_from_captions(r["references"], OBJECTS, INV) | inst.get(fn2id.get(r["image"]), set())
      for r in items}


class Clip:
    """CLIP ViT-B/32 for the retrieval corpus and S_retr: L2-normalised image/caption vectors, stored as float16."""
    def __init__(self, name="openai/clip-vit-base-patch32"):
        from transformers import CLIPModel, CLIPProcessor
        self.device = DEVICE
        self.model = CLIPModel.from_pretrained(name).to(self.device).eval()
        self.proc = CLIPProcessor.from_pretrained(name)

    @torch.no_grad()
    def _encode(self, fn, batches):
        out = []
        for inp in batches:
            with torch.autocast(self.device, enabled=self.device == "cuda"):
                f = fn(**inp.to(self.device))
            f = f if isinstance(f, torch.Tensor) else f.pooler_output   # transformers >= 5 returns an object; pooler_output is the projected vector (as in clipscore.py)
            out.append(torch.nn.functional.normalize(f.float(), dim=-1).cpu().numpy().astype(np.float16))
        return np.concatenate(out)

    def text(self, texts, bs=512):
        return self._encode(self.model.get_text_features, (self.proc(text=texts[i: i + bs], return_tensors="pt", padding=True,
                                                                     truncation=True) for i in range(0, len(texts), bs)))

    def images(self, paths, bs=64):
        return self._encode(self.model.get_image_features, (self.proc(images=[Image.open(p).convert("RGB") for p in paths[i: i + bs]],
                                                                      return_tensors="pt") for i in range(0, len(paths), bs)))


def minmax(v):
    """Min-max normalisation over the candidates of one image; returns 0.5 when all values are equal."""
    lo, hi = v.min(), v.max()
    return np.full_like(v, 0.5) if hi - lo < 1e-9 else (v - lo) / (hi - lo)


def objs_of(text):
    """List of COCO objects (mapped to class names) mentioned in a caption, as in CHAIR."""
    return chair_metric.caption_to_words(text, OBJECTS, INV)[1]


def score(captions, meteor=USE_METEOR, spice=False):
    """BLEU-1..4, METEOR, ROUGE-L, CIDEr (x100), optional SPICE, CHAIRs/CHAIRi; 'simple' tokenisation as in the paper."""
    refs = [r["references"] for r in items]
    m = compute_metrics(captions, refs, use_meteor=meteor, use_cider=True, use_spice=spice, tokenizer="simple")
    ch = chair_metric.compute_chair({r["image"]: c for r, c in zip(items, captions)}, GT, OBJECTS, INV)
    out = {k: round(float(v) * (100 if k == "CIDEr" else 1), 2) for k, v in m.items()}
    out.update(CHAIRs=round(ch["CHAIRs"], 2), CHAIRi=round(ch["CHAIRi"], 2),
               mean_len=round(float(np.mean([len(c.split()) for c in captions])), 2))
    return out


print(f"{len(items)} {SPLIT} images | device {DEVICE} | checkpoint {CHECKPOINT.name} | δ = {DELTA:g}")

In [ ]:
# ==========================================
# OBJECT EVIDENCE E(I) = YOLO-World  U  {objects named by >= MAJ of the K retrieved captions}
# ==========================================
# Retrieval corpus: all training captions (train + restval), encoded with CLIP ViT-B/32 and cached.
clip32 = Clip()
caps_f, emb_f = RETRIEVAL_CACHE / "train_captions.json", RETRIEVAL_CACHE / "train_caption_emb_b32.npy"
if caps_f.is_file() and emb_f.is_file():
    BANK_CAPS, BANK_EMB = json.loads(caps_f.read_text(encoding="utf-8")), np.load(emb_f)
else:
    BANK_CAPS = [s["raw"].strip().lower() for im in data["images"] if im["split"] in ("train", "restval") for s in im["sentences"]]
    print(f"Encoding the retrieval corpus: {len(BANK_CAPS)} captions (done once)")
    BANK_EMB = clip32.text(BANK_CAPS)
    RETRIEVAL_CACHE.mkdir(parents=True, exist_ok=True)
    caps_f.write_text(json.dumps(BANK_CAPS), encoding="utf-8")
    np.save(emb_f, BANK_EMB)

# K captions closest to the image by image-caption cosine (CLIP B/32); the corpus has no val/test captions, so no label leakage.
img_emb = clip32.images(paths)
dt = torch.float16 if clip32.device == "cuda" else torch.float32
bank_t = torch.from_numpy(BANK_EMB).to(clip32.device, dt)
NB = np.concatenate([(torch.from_numpy(img_emb[i: i + 256]).to(clip32.device, dt) @ bank_t.T).topk(K_RETR, dim=1).indices.cpu().numpy()
                     for i in range(0, len(img_emb), 256)])
del bank_t

extractor = YoloObjectExtractor(str(YOLO_MODEL), YOLO_CONF, YOLO_MAX_OBJECTS)
vocab_mod.set_object_noun_vocab(extractor.class_names())   # object nouns for the negative evidence of S_det
YOLO_OBJS = [extractor.extract(p) for p in tqdm(paths, desc="YOLO-World")]
YOLO_SET = [{INV[o.label] for o in objs if o.confidence >= YOLO_CONF and o.label in INV} for objs in YOLO_OBJS]
NB_COUNTS = [Counter(o for j in idx for o in set(objs_of(BANK_CAPS[j]))) for idx in NB]   # number of retrieved captions naming each object
RETR_SET = [{o for o, n in c.items() if n >= MAJ} for c in NB_COUNTS]   # objects named by >= MAJ retrieved captions
EVIDENCE = [y | r for y, r in zip(YOLO_SET, RETR_SET)]
EVIDENCE_OF = {"union": EVIDENCE, "yolo": YOLO_SET, "retr": RETR_SET}   # evidence source: Y+R, Y only (C8), R only (C9)
print(f"Average {np.mean([len(e) for e in EVIDENCE]):.2f} objects in E(I) per image")

In [ ]:
# ==========================================
# FIXED BLOCKING RULE ON THE LOGITS:  z'(w) = z(w) - δ · 1[w is a COCO object word, o(w) not in E(I)]
# ==========================================
# Applied at every decoding step of both plain beam search and diverse beam search; non-object words are unchanged.
IRREGULAR = {"men": "man", "women": "woman", "kids": "kid", "guys": "guy", "mice": "mouse"}
VOCAB = json.loads(VOCAB_FILE.read_text(encoding="utf-8"))
TOK2OBJ = {}
for word, idx in VOCAB.items():
    w = IRREGULAR.get(word) or chair_metric._LEMMATIZER.lemmatize(word, pos="n")
    if w in OBJECTS:
        TOK2OBJ[idx] = INV[w]
print(f"{len(TOK2OBJ)} vocabulary tokens map to COCO objects")

CURRENT_BIAS = None
if not hasattr(local_captioner, "_ORIG_DECODE_STEP"):
    local_captioner._ORIG_DECODE_STEP = local_captioner.ImageCaptioningTransformer.decode_step


def _biased_decode_step(self, memory, sequence):
    logits = local_captioner._ORIG_DECODE_STEP(self, memory, sequence)
    return logits if CURRENT_BIAS is None else logits + CURRENT_BIAS


local_captioner.ImageCaptioningTransformer.decode_step = _biased_decode_step


def rule_bias(evidence, delta=None):
    delta = DELTA if delta is None else delta
    if not delta:
        return None
    bias = torch.zeros(len(VOCAB))
    for tid, obj in TOK2OBJ.items():
        if obj not in evidence:
            bias[tid] = -delta
    return bias.to(DEVICE)

In [ ]:
# ==========================================
# GENERATE 5 CANDIDATES (with blocking) AND SCORE THE THREE SIGNALS
# ==========================================
from clipscore import ClipScorer
from formula_reranker import FormulaReranker
from generate import CandidateCaptionGenerator

generator = CandidateCaptionGenerator(checkpoint_path=CHECKPOINT, vocab_path=VOCAB_FILE, max_candidates=5, num_beams=10,
                                      num_beam_groups=5, diversity_penalty=0.8, max_new_tokens=MAX_NEW_TOKENS, safe_beam_size=3)
clip16 = ClipScorer(model_name=CLIP_SCORER)
formula = FormulaReranker(alpha=W_BASE, beta=W_ALIGN, gamma=W_DET, formula_mode="linear", clip_scorer=clip16,
                          belief_neg_confidence=0.0)

RUN_INFO = {"sobj_conf": SOBJ_CONF, "w_retr": W_RETR, "w_obj": W_OBJ}   # only changes caption selection, no need to regenerate


def run_key(run):
    key = {"checkpoint": str(CHECKPOINT), "split": SPLIT, "start": START, "images": len(items), "delta": run[0],
           "maj": MAJ, "k_retr": K_RETR, "yolo_model": YOLO_MODEL.name, "weights": [W_BASE, W_ALIGN, W_DET]}
    if run[1] != "union":   # the Y+R file keeps the old key so existing files can be resumed
        key["evidence"] = run[1]
    return key


def save(run, rows, metrics=None):
    out = output_of(run)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps({"run": run_key(run), "rerank": RUN_INFO, "metrics": metrics, "rows": rows}, ensure_ascii=False),
                   encoding="utf-8")


def generate_rows(run):
    """Generate 5 candidates with the run's blocking rule (delta, evidence source) and score the three signals; saves progress and can resume."""
    global CURRENT_BIAS   # _biased_decode_step reads this global variable
    delta, src = run
    rows, out = [], output_of(run)
    if out.is_file():
        old = json.loads(out.read_text(encoding="utf-8"))
        if old.get("run") == run_key(run):
            rows = old["rows"]
            print(f"{label(run)}: resuming from image {len(rows)}")
    t0, done0 = time.time(), len(rows)
    for i in tqdm(range(done0, len(items)), desc=f"Generating and scoring candidates, {label(run)}"):
        r, path, objs = items[i], paths[i], YOLO_OBJS[i]
        CURRENT_BIAS = rule_bias(EVIDENCE_OF[src][i], delta)
        cands = generator.generate(path, objs)
        CURRENT_BIAS = None
        scored = clip16.score(path, cands)
        ranked = formula.rerank(scored, objs)
        rows.append({
            "image": r["image"], "references": r["references"],
            "baseline_candidate": cands[0].text, "caption": ranked[0].caption.text,
            "objects": [{"label": o.label, "confidence": o.confidence, "box_xyxy": list(o.box_xyxy) if o.box_xyxy else None} for o in objs],
            "candidates": [c.__dict__ for c in scored],
            "ranking": [{"caption": x.caption.text, "rank": x.rank, "rerank_score": x.rerank_score, "reason": x.reason} for x in ranked],
            "neighbours": [int(j) for j in NB[i]], "evidence": sorted(EVIDENCE_OF[src][i]),
        })
        if (i + 1) % SAVE_EVERY == 0:
            save(run, rows)
            sec = (time.time() - t0) / (i + 1 - done0)   # tqdm does not show in the log under Save & Run All, so print this line too
            print(f"{label(run)}: {i + 1}/{len(items)} images | {sec:.2f} s/image | about {sec * (len(items) - i - 1) / 60:.0f} min left",
                  flush=True)
    CURRENT_BIAS = None
    save(run, rows)
    if len(rows) > done0:
        print(f"{label(run)}: {(time.time() - t0) / (len(rows) - done0):.2f} seconds per image")
    return rows


ROWS_BY_RUN = {run: generate_rows(run) for run in RUNS}

In [ ]:
# ==========================================
# THE TWO PROPOSED SIGNALS AND THE CAPTION SELECTION VARIANTS
# ==========================================
# S_retr(c) = mean cosine between caption c and the K retrieved captions (CLIP ViT-B/32)
# S_obj(c)  = -(number of COCO objects that c names but that are not in E(I)); E(I) uses threshold r (MAJ or 1)
texts = sorted({c["text"] for rows in ROWS_BY_RUN.values() for row in rows for c in row["candidates"]})
TEXT_EMB = dict(zip(texts, clip32.text(texts)))
YOLO_SET_SOBJ = [{INV[o.label] for o in objs if o.confidence >= SOBJ_CONF and o.label in INV} for objs in YOLO_OBJS]


def selections(rows, src):
    per_image = []
    for row, idx, ysup, cnt in zip(rows, NB, YOLO_SET_SOBJ, NB_COUNTS):
        cands = [c["text"] for c in row["candidates"]]
        rs = {x["caption"]: x["rerank_score"] for x in row["ranking"]}
        base = np.array([rs[t] for t in cands], dtype=np.float64)          # three signals (normalised within the image)
        ce = np.stack([TEXT_EMB[t] for t in cands]).astype(np.float32)
        s_retr = (ce @ BANK_EMB[idx].astype(np.float32).T).mean(axis=1)
        s_obj = {}
        for thr in sorted({MAJ, 1}):
            ret = {o for o, n in cnt.items() if n >= thr}
            sup = {"union": ysup | ret, "yolo": ysup, "retr": ret}[src]
            s_obj[thr] = np.array([-sum(o not in sup for o in objs_of(t)) for t in cands], dtype=np.float64)
        per_image.append({"cands": cands, "base": base, "s_retr": s_retr, "s_obj": s_obj})

    def pick(fn):
        return [p["cands"][int(np.argmax(fn(p)))] for p in per_image]

    sel = {
        "before_rerank": [row["baseline_candidate"] for row in rows],
        "three_score": pick(lambda p: p["base"]),
        "plus_s_retr": pick(lambda p: p["base"] + W_RETR * minmax(p["s_retr"])),
        "plus_s_obj": pick(lambda p: p["base"] + W_OBJ * minmax(p["s_obj"][MAJ])),
        "five_score": pick(lambda p: p["base"] + W_RETR * minmax(p["s_retr"]) + W_OBJ * minmax(p["s_obj"][MAJ])),
        "five_score_1of5": pick(lambda p: p["base"] + W_RETR * minmax(p["s_retr"]) + W_OBJ * minmax(p["s_obj"][1])),
    }
    equal = round(100 * float(np.mean([p["s_obj"][MAJ].max() == p["s_obj"][MAJ].min() for p in per_image])), 1)
    return sel, equal


SELECTIONS = {run: selections(rows, run[1]) for run, rows in ROWS_BY_RUN.items()}
for run, (_, equal) in SELECTIONS.items():
    print(f"{label(run)}: S_obj is equal for all five candidates (ranking unchanged) on {equal}% of images")

In [ ]:
# ==========================================
# SCORING AND SAVING (one file per run)
# ==========================================
SPICE_ON = {"before_rerank", "three_score", "five_score"}
KEYS = ("BLEU-4", "METEOR", "ROUGE-L", "CIDEr", "CHAIRs", "CHAIRi")
METRICS = {}
for run, (sel, equal) in SELECTIONS.items():
    print(f"===== {label(run)}" + (" (no blocking)" if not run[0] else " (with blocking)"))
    METRICS[run] = {}
    for name, caps in sel.items():
        METRICS[run][name] = score(caps, spice=COMPUTE_SPICE and name in SPICE_ON)
        print(f"{name:16s}", {k: METRICS[run][name].get(k) for k in KEYS})
    METRICS[run]["sobj_equal_pct"] = equal
    rows = ROWS_BY_RUN[run]
    for row, cap in zip(rows, sel["five_score"]):
        row["caption_five_score"] = cap
    save(run, rows, METRICS[run])
    print(f"Saved {output_of(run)}")